# 06 - Model Comparison

> Load all 4 model predictions on the same test set (2014), compute unified metrics, and select the best model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
COLORS = {'ARIMA':'#BF212F','Prophet':'#003B73','XGBoost':'#0074B7','Random Forest':'#27AE60'}
print("Libraries loaded")

## 1. Load All Predictions

In [ ]:
arima = pd.read_csv('../predictions/arima_predictions.csv', parse_dates=['ds'])
prophet = pd.read_csv('../predictions/prophet_predictions.csv', parse_dates=['ds'])
xgb = pd.read_csv('../predictions/xgboost_predictions.csv', parse_dates=['ds'])
rf = pd.read_csv('../predictions/random_forest_predictions.csv', parse_dates=['ds'])
print(f"Loaded predictions: ARIMA({len(arima)}), Prophet({len(prophet)}), XGBoost({len(xgb)}), RF({len(rf)})")

## 2. Calculate Metrics

In [ ]:
def calc(name, y_true, y_pred):
    return {'Model': name,
            'MAPE (%)': np.mean(np.abs((y_true - y_pred) / y_true)) * 100,
            'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'R2': r2_score(y_true, y_pred)}

metrics = pd.DataFrame([
    calc('ARIMA', arima['actual'].values, arima['predicted'].values),
    calc('Prophet', prophet['actual'].values, prophet['predicted'].values),
    calc('XGBoost', xgb['actual'].values, xgb['predicted'].values),
    calc('Random Forest', rf['actual'].values, rf['predicted'].values)
]).sort_values('MAPE (%)').reset_index(drop=True)

print("=" * 70)
print("MODEL COMPARISON (Test Set: Jan 2014 - Dec 2014)")
print("=" * 70)
print(metrics.round(2).to_string(index=False))
print(f"\nBest Model: {metrics.iloc[0]['Model']} (MAPE: {metrics.iloc[0]['MAPE (%)']:.2f}%)")
print(f"Target MAPE < 15%: {'ACHIEVED' if metrics.iloc[0]['MAPE (%)'] < 15 else 'NOT MET'}")

metrics.to_csv('../evaluation/model_comparison.csv', index=False)
print("\nSaved: evaluation/model_comparison.csv")

## 3. Visualize Metrics Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
bar_colors = [COLORS.get(m, '#95A5A6') for m in metrics['Model']]

for ax, col, title in zip(axes.flat,
    ['MAPE (%)','MAE','RMSE','R2'],
    ['MAPE (Lower = Better)','MAE (Lower = Better)','RMSE (Lower = Better)','R2 Score (Higher = Better)']):
    bars = ax.bar(metrics['Model'], metrics[col], color=bar_colors, edgecolor='white', width=0.6)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel(col)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{bar.get_height():.2f}',
                ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.suptitle('Forecast Model Evaluation Metrics', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../visualizations/model_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. Overlay All Predictions

In [ ]:
train_ts = pd.read_csv('../data/train_ts.csv', parse_dates=['ds'])

plt.figure(figsize=(16, 7))
plt.plot(train_ts['ds'], train_ts['y'], color='gray', linewidth=1.5, alpha=0.5, label='Train')
plt.plot(arima['ds'], arima['actual'], color='black', linewidth=2, marker='o', markersize=5, label='Actual (Test)')
plt.plot(arima['ds'], arima['predicted'], '--s', color=COLORS['ARIMA'], linewidth=1.5, markersize=5, label='ARIMA')
plt.plot(prophet['ds'], prophet['predicted'], '--d', color=COLORS['Prophet'], linewidth=1.5, markersize=5, label='Prophet')
plt.plot(xgb['ds'], xgb['predicted'], '--x', color=COLORS['XGBoost'], linewidth=1.5, markersize=6, label='XGBoost')
plt.plot(rf['ds'], rf['predicted'], '--^', color=COLORS['Random Forest'], linewidth=1.5, markersize=5, label='RF')

plt.axvline(pd.Timestamp('2014-01-01'), color='red', linestyle=':', alpha=0.6, label='Train/Test Split')
plt.title('All Models vs Actuals (Test Set 2014)', fontsize=16, fontweight='bold')
plt.xlabel('Date'); plt.ylabel('Monthly Sales ($)'); plt.legend(loc='upper left')
plt.tight_layout()
plt.savefig('../visualizations/all_models_overlay.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Key Findings

| # | Finding |
|---|---|
| 1 | **Prophet** achieves the lowest MAPE, excelling at handling yearly seasonality |
| 2 | Tree-based models (XGBoost, RF) struggle with only 48 monthly data points |
| 3 | **ARIMA** provides a solid statistical baseline but lacks Prophet's flexibility |
| 4 | All predictions correctly capture the **Q4 seasonal spike** in sales |

Proceed to **07_Final_Forecast.ipynb** to retrain the best model on full data and generate the 12-month forecast.